# Phase 11 — Overfitting & Underfitting Diagnosis
**ChurnOps MLOps Pipeline**

---

### Objectives
1. **Generalization Gap Evaluation**: Quantify train-val gaps across all 4 baseline models against `gap_threshold: 0.08`.
2. **Model Complexity Curves**: Plot validation curves across hyperparameters (`max_depth` for RF and XGB, `C` for Logistic Regression) to pinpoint the exact root causes of variance explosion.
3. **Out-of-Fold Slice Error Analysis**: Evaluate out-of-fold validation predictions across business cohorts (Contract, Internet Service, Tenure, Payment Method) to detect vulnerable customer segments.
4. **Prediction Confidence & Error Dissection**: Profile probability distributions for True Positives, True Negatives, False Alarms (FP), and Missed Churners (FN).
5. **Actionable Regularization Prescriptions**: Define exact hyperparameter search spaces and regularization tactics for Phase 12 (Hyperparameter Optimization).

> **Data Leakage Rule (R3)**:
> All evaluations and curves are derived strictly from `data/processed/train.csv` (5,634 rows) across the pre-partitioned 5 stratified folds.
> The holdout test set (`test.csv`) remains completely untouched.


In [1]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, HTML

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
)

from src.data.preprocessing import build_full_pipeline
from src.utils.config import PARAMS

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
print("Environment and dependencies initialized successfully.")


Environment and dependencies initialized successfully.


---
## 1. Load Data & Cross-Validation Folds
We load `train.csv` (5,634 rows) and the exact 5-fold CV splits (`cv_folds.json`).


In [2]:
train_df = pd.read_csv("../data/processed/train.csv")
with open("../data/processed/cv_folds.json", "r") as f:
    cv_folds = json.load(f)

TARGET = PARAMS["dataset"]["target_column"]
X = train_df.drop(columns=[TARGET])
y = train_df[TARGET]

print(f"Loaded train.csv: {train_df.shape[0]} samples, {train_df.shape[1]} features")
print(f"5-Fold CV partitions loaded: {len(cv_folds)} folds, ~{len(cv_folds['fold_1']['val_indices'])} validation samples per fold")


Loaded train.csv: 5634 samples, 20 features
5-Fold CV partitions loaded: 5 folds, ~1127 validation samples per fold


---
## 2. Generalization Gap & Bias-Variance Summary
Evaluating the gap $\Delta = \text{Train F1} - \text{Val F1}$ against the project threshold (`gap_threshold = 0.08`).


In [3]:
# Load Phase 10 / 11 results report
report_path = Path("../reports/diagnosis_report.json")
if not report_path.exists():
    from src.models.diagnose import run_diagnosis
    diag_rep = run_diagnosis()
else:
    with open(report_path, "r") as f:
        diag_rep = json.load(f)

summary_rows = []
for model_name, diag in diag_rep["summary_diagnosis"].items():
    summary_rows.append({
        "Model": model_name,
        "Status": diag["status"],
        "Gap (Train F1 - Val F1)": f"{diag['gap']:+.4f}",
        "Exceeds 0.08 Gate?": "YES (Severe Overfit)" if diag['gap'] > 0.08 else "NO (Passed Gate)",
        "Prescribed Direction": diag["action"]
    })

summary_df = pd.DataFrame(summary_rows)
display(HTML(summary_df.to_html(index=False)))


Model,Status,Gap (Train F1 - Val F1),Exceeds 0.08 Gate?,Prescribed Direction
DummyClassifier,UNDERFITTING,-0.0218,NO (Passed Gate),Benchmark only
LogisticRegression,GOOD_FIT,+0.0067,NO (Passed Gate),Optimize C and threshold
RandomForest,SEVERE_OVERFITTING,+0.4494,YES (Severe Overfit),"Prune max_depth to 6-8, increase min_samples_leaf"
XGBoost,SEVERE_OVERFITTING,+0.3788,YES (Severe Overfit),"Prune max_depth to 2-3, add subsampling & L1/L2"


---
## 3. Model Complexity Curves (Validation Curves)
To understand *why* Random Forest and XGBoost overfit severely while Logistic Regression remains stable, we plot **Validation Complexity Curves**:
- **RandomForest**: Train and validation F1 vs. `max_depth` $\in [2, 4, 6, 8, 12, 16, \text{None}]$.
- **XGBoost**: Train and validation F1 vs. `max_depth` $\in [1, 2, 3, 4, 6, 8]$.
- **LogisticRegression**: Train and validation F1 vs. $C \in [0.001, 0.01, 0.1, 1.0, 10.0]$.

This reveals the exact parameter regimes where high variance triggers.


In [4]:
from src.models.diagnose import compute_complexity_curves

curves = compute_complexity_curves(X, y, cv_folds)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: RandomForest Depth
ax1 = axes[0]
c_rf = curves["RandomForest_depth"]
x1 = np.arange(len(c_rf["param_values"]))
ax1.plot(x1, c_rf["train_f1"], "o--", color="#10B981", label="Train F1", lw=2)
ax1.plot(x1, c_rf["val_f1"], "s-", color="#047857", label="Validation F1", lw=2.5)
ax1.fill_between(x1, c_rf["val_f1"], c_rf["train_f1"], color="#10B981", alpha=0.15, label="Generalization Gap")
ax1.axhline(0.60, color="gray", ls=":", label="Min Gate (0.60)")
ax1.set_xticks(x1)
ax1.set_xticklabels(c_rf["param_values"])
ax1.set_title("RandomForest: max_depth Complexity Curve", fontweight="bold", fontsize=11)
ax1.set_xlabel("max_depth", fontsize=10)
ax1.set_ylabel("F1 Score", fontsize=10)
ax1.legend(loc="upper left", fontsize=8.5)
ax1.grid(alpha=0.3)
ax1.set_ylim(0.40, 1.02)

# Panel 2: XGBoost Depth
ax2 = axes[1]
c_xgb = curves["XGBoost_depth"]
x2 = np.arange(len(c_xgb["param_values"]))
ax2.plot(x2, c_xgb["train_f1"], "o--", color="#F59E0B", label="Train F1", lw=2)
ax2.plot(x2, c_xgb["val_f1"], "s-", color="#B45309", label="Validation F1", lw=2.5)
ax2.fill_between(x2, c_xgb["val_f1"], c_xgb["train_f1"], color="#F59E0B", alpha=0.15, label="Generalization Gap")
ax2.axhline(0.60, color="gray", ls=":", label="Min Gate (0.60)")
ax2.set_xticks(x2)
ax2.set_xticklabels(c_xgb["param_values"])
ax2.set_title("XGBoost: max_depth Complexity Curve", fontweight="bold", fontsize=11)
ax2.set_xlabel("max_depth", fontsize=10)
ax2.legend(loc="upper left", fontsize=8.5)
ax2.grid(alpha=0.3)
ax2.set_ylim(0.40, 1.02)

# Panel 3: LogisticRegression C
ax3 = axes[2]
c_lr = curves["LogisticRegression_C"]
x3 = np.arange(len(c_lr["param_values"]))
ax3.plot(x3, c_lr["train_f1"], "o--", color="#3B82F6", label="Train F1", lw=2)
ax3.plot(x3, c_lr["val_f1"], "s-", color="#1D4ED8", label="Validation F1", lw=2.5)
ax3.fill_between(x3, c_lr["val_f1"], c_lr["train_f1"], color="#3B82F6", alpha=0.15, label="Generalization Gap")
ax3.axhline(0.60, color="gray", ls=":", label="Min Gate (0.60)")
ax3.set_xticks(x3)
ax3.set_xticklabels(c_lr["param_values"])
ax3.set_title("LogisticRegression: C Regularization Curve", fontweight="bold", fontsize=11)
ax3.set_xlabel("C (Inverse Regularization)", fontsize=10)
ax3.legend(loc="lower right", fontsize=8.5)
ax3.grid(alpha=0.3)
ax3.set_ylim(0.55, 0.70)

plt.suptitle("Validation Complexity Curves (Generalization Gap Diagnosis)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


[2026-10-06 11:55:50] INFO     diagnose - Computing complexity curves...


[2026-10-06 11:56:17] INFO     diagnose - Complexity curves calculated successfully.


C:\Users\Abhishek\AppData\Local\Temp\ipykernel_42900\2603258701.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 4. Slice / Cohort Error Diagnostics (Out-of-Fold Analysis)
To identify which business segments suffer disproportionate misclassifications, we examine Out-of-Fold (OOF) cross-validation predictions across:
- **Contract Type**: `Month-to-month` vs. `One year` vs. `Two year`
- **Internet Service**: `Fiber optic` vs. `DSL` vs. `No internet`
- **Tenure Group**: `0-12 months`, `13-24 months`, `25-48 months`, `49-72 months`


In [5]:
from src.models.diagnose import generate_oof_predictions, compute_slice_diagnostics

oof_data = generate_oof_predictions(X, y, cv_folds)
slice_rep = compute_slice_diagnostics(train_df, y, oof_data)

# Tabulate Contract cohort performance
contract_rows = []
for contract_type in ["Month-to-month", "One year", "Two year"]:
    c_info = slice_rep["Contract"][contract_type]
    m_info = c_info["models"]["LogisticRegression"]
    contract_rows.append({
        "Contract Cohort": contract_type,
        "Cohort Size": c_info["sample_count"],
        "Actual Churn Rate": f"{c_info['churn_rate'] * 100:.1f}%",
        "LR F1 Score": m_info["f1"],
        "LR Recall": m_info["recall"],
        "LR Precision": m_info["precision"],
        "False Positives": m_info["fp"],
        "False Negatives": m_info["fn"]
    })

display(HTML("<h4>Performance Breakdown by Contract Type (Logistic Regression)</h4>"))
display(HTML(pd.DataFrame(contract_rows).to_html(index=False)))


[2026-10-06 11:56:17] INFO     diagnose - Generating Out-Of-Fold predictions for all models...


[2026-10-06 11:56:21] INFO     diagnose - OOF predictions generated successfully.


[2026-10-06 11:56:21] INFO     diagnose - Computing slice error diagnostics...


Contract Cohort,Cohort Size,Actual Churn Rate,LR F1 Score,LR Recall,LR Precision,False Positives,False Negatives
Month-to-month,3102,42.8%,0.6595,0.8793,0.5276,1044,160
One year,1173,11.1%,0.2000,0.1692,0.2444,68,108
Two year,1359,2.9%,0.0000,0.0000,0.0000,0,39


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

slices_to_plot = [
    ("Contract", "Performance across Contract Types"),
    ("InternetService", "Performance across Internet Services"),
    ("tenure_group", "Performance across Customer Tenure"),
]

for ax, (slice_feat, title) in zip(axes, slices_to_plot):
    data = slice_rep.get(slice_feat, {})
    categories = list(data.keys())
    x = np.arange(len(categories))
    width = 0.25
    models = ["LogisticRegression", "RandomForest", "XGBoost"]
    colors = ["#3B82F6", "#10B981", "#F59E0B"]

    for idx, (m, c) in enumerate(zip(models, colors)):
        f1s = [data[cat]["models"][m]["f1"] for cat in categories]
        rects = ax.bar(x + (idx - 1) * width, f1s, width, label=m, color=c, alpha=0.85)
        for rect, f1 in zip(rects, f1s):
            if f1 > 0:
                ax.text(rect.get_x() + rect.get_width() / 2, f1 + 0.015,
                        f"{f1:.2f}", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels([c.replace(" ", "\n") for c in categories], fontsize=9)
    ax.set_title(title, fontweight="bold", fontsize=11)
    ax.set_ylabel("F1 Score", fontsize=10)
    ax.set_ylim(0, 0.85)
    ax.grid(alpha=0.3, axis="y")
    ax.legend(fontsize=8.5, loc="upper right")

plt.suptitle("Out-of-Fold F1 Breakdown across Customer Cohorts", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


C:\Users\Abhishek\AppData\Local\Temp\ipykernel_42900\4046678484.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 5. Prediction Confidence & Confusion Dissection
We inspect the distribution of predicted probabilities $P(\text{Churn}=1)$ partitioned into:
- **True Positives (TP)**: Correctly detected churners.
- **True Negatives (TN)**: Correctly identified retained customers.
- **False Positives (FP)**: False alarms (wasted retention spend).
- **False Negatives (FN)**: Missed departures (unmitigated revenue loss).


In [7]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models = ["LogisticRegression", "RandomForest", "XGBoost"]

for ax, m in zip(axes, models):
    preds = oof_data[m]["pred"]
    probs = oof_data[m]["proba"]
    y_arr = y.values

    tp_probs = probs[(y_arr == 1) & (preds == 1)]
    tn_probs = probs[(y_arr == 0) & (preds == 0)]
    fp_probs = probs[(y_arr == 0) & (preds == 1)]
    fn_probs = probs[(y_arr == 1) & (preds == 0)]

    bp = ax.boxplot(
        [tn_probs, fp_probs, fn_probs, tp_probs],
        tick_labels=["TN\n(Loyal)", "FP\n(False Alarm)", "FN\n(Missed)", "TP\n(Detected)"],
        patch_artist=True,
        medianprops=dict(color="black", lw=1.5),
    )

    palette = ["#93C5FD", "#FCA5A5", "#FDE047", "#86EFAC"]
    for patch, col in zip(bp["boxes"], palette):
        patch.set_facecolor(col)

    ax.axhline(0.5, color="red", ls="--", alpha=0.7, label="Default Threshold (0.50)")
    ax.set_title(f"{m}: Predicted Probabilities", fontweight="bold", fontsize=11)
    ax.set_ylabel("Predicted P(Churn=1)", fontsize=10)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(alpha=0.3, axis="y")
    ax.legend(loc="upper left", fontsize=8.5)

plt.suptitle("Probability Calibration across Confusion Quadrants", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


C:\Users\Abhishek\AppData\Local\Temp\ipykernel_42900\3040537075.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 6. Actionable Regularization Prescriptions for Phase 12 HPO
Based on the empirical complexity curves and slice error rates, we formulate explicit search constraints:

| Model | Primary Vulnerability | Prescribed Regularization Space | Target Gap |
|---|---|---|---|
| **RandomForest** | Unconstrained tree depth (`max_depth=None`, gap = 0.45) | `max_depth: [4, 6, 8]`, `min_samples_leaf: [4, 8, 16]`, `max_features: ['sqrt', 0.5]` | $\le 0.06$ |
| **XGBoost** | Deep boosting trees (`max_depth=6`, gap = 0.38) | `max_depth: [2, 3, 4]`, `learning_rate: [0.03, 0.05, 0.1]`, `subsample: [0.6, 0.8]`, `reg_alpha: [0.1, 1.0]` | $\le 0.06$ |
| **LogisticRegression** | Linear capacity boundary (good fit, gap = 0.007) | `C: [0.01, 0.05, 0.1, 1.0]`, optimize decision threshold to $\sim 0.60$ | $\le 0.02$ |


In [8]:
prescriptions = diag_rep["prescriptions"]
for model_name, p in prescriptions.items():
    if model_name == "DummyClassifier":
        continue
    print(f"==================================================")
    print(f"Model: {model_name} ({p['diagnosis']})")
    print(f"  Vulnerability: {p['primary_vulnerability']}")
    print(f"  Finding:       {p['complexity_curve_finding']}")
    print(f"  Strategy:      {p['regularization_strategy']}")
    print(f"  Phase 12 Search Space:")
    for param, values in p["phase_12_search_space"].items():
        print(f"    {param}: {values}")


Model: RandomForest (SEVERE_OVERFITTING)
  Vulnerability: Unconstrained tree depth grows trees until leaves are pure (Train F1 ~0.996), memorizing noise.
  Finding:       Restricting max_depth to 6-8 drops generalization gap from 0.45 down to 0.02-0.06 while boosting validation F1 from 0.54 to 0.65.
  Strategy:      Aggressive pruning via max_depth and min_samples_leaf to force tree generalization.
  Phase 12 Search Space:
    model__max_depth: [4, 5, 6, 8, 10]
    model__min_samples_leaf: [2, 4, 8, 16]
    model__min_samples_split: [5, 10, 20]
    model__max_features: ['sqrt', 'log2', 0.4, 0.6]
    model__n_estimators: [100, 200, 300]
Model: XGBoost (SEVERE_OVERFITTING)
  Vulnerability: Default max_depth=6 with 200 boosting iterations overfits dense one-hot encoded interactions.
  Finding:       max_depth=2-3 achieves optimal validation F1 (~0.64) with gap < 0.07. Depths >= 4 trigger steep overfitting.
  Strategy:      Shallow trees (max_depth 2-4) + stochastic subsampling (row/col) +